In [ ]:
!pip install -q streamlit tavily-python chromadb python-dotenv

!pip install -q -U "google-genai>=2.0.0"

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your .env file when prompted

from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())
print("Tavily key loaded:", bool(os.getenv("TAVILY_API_KEY")))
print("Gemini key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

In [ ]:
%%writefile app.py
import os
import uuid
import datetime

import streamlit as st
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types
from tavily import TavilyClient
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

load_dotenv(find_dotenv())

GEMINI_MODEL = "gemini-3.5-flash"
EMBED_MODEL = "gemini-embedding-001"

st.set_page_config(page_title="Smart Research Assistant", page_icon="🔎", layout="wide")


@st.cache_resource
def get_genai_client():
    return genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))


@st.cache_resource
def get_tavily_client():
    return TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, task_type="RETRIEVAL_DOCUMENT"):
        self.task_type = task_type

    def __call__(self, input: Documents) -> Embeddings:
        client = get_genai_client()
        result = client.models.embed_content(
            model=EMBED_MODEL,
            contents=input,
            config=types.EmbedContentConfig(task_type=self.task_type),
        )
        return [e.values for e in result.embeddings]


@st.cache_resource
def get_collection():
    chroma_client = chromadb.EphemeralClient()
    return chroma_client.get_or_create_collection(
        name="research_memory",
        embedding_function=GeminiEmbeddingFunction(task_type="RETRIEVAL_DOCUMENT"),
    )


def embed_query(text):
    client = get_genai_client()
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=[text],
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return result.embeddings[0].values


def web_search(query, max_results=5):
    tavily_client = get_tavily_client()
    response = tavily_client.search(query=query, search_depth="advanced", max_results=max_results)
    return response.get("results", [])


def summarize_with_gemini(query, search_results):
    client = get_genai_client()
    context = "\n\n".join(
        f"Source: {r.get('url')}\nTitle: {r.get('title')}\nContent: {r.get('content')}"
        for r in search_results
    )
    prompt = (
        "You are a research assistant. Using the web search results below, write a clear, "
        "well-organized summary that answers the user's question. Be factual and specific.\n\n"
        f"Question: {query}\n\nSearch Results:\n{context}\n\n"
        "Write a concise summary (3-6 paragraphs)."
    )
    interaction = client.interactions.create(model=GEMINI_MODEL, input=prompt)
    return interaction.output_text


def store_research(query, summary, sources):
    collection = get_collection()
    doc_id = str(uuid.uuid4())
    collection.add(
        documents=[summary],
        metadatas=[{
            "query": query,
            "sources": ", ".join(sources),
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        }],
        ids=[doc_id],
    )


def answer_followup(question, n_results=3):
    client = get_genai_client()
    collection = get_collection()
    query_emb = embed_query(question)
    results = collection.query(query_embeddings=[query_emb], n_results=n_results)

    docs = results["documents"][0] if results["documents"] else []
    retrieved_context = "\n\n---\n\n".join(docs) if docs else "No relevant stored research found yet."

    prompt = (
        "You are a research assistant. Answer the follow-up question using ONLY the context "
        "below, which comes from earlier research sessions. If the context is insufficient, "
        "say so clearly instead of guessing.\n\n"
        f"Context:\n{retrieved_context}\n\nFollow-up question: {question}\n\nAnswer:"
    )
    interaction = client.interactions.create(model=GEMINI_MODEL, input=prompt)
    return interaction.output_text


# ---------------- UI ----------------

if "history" not in st.session_state:
    st.session_state.history = []

st.title("🔎 Smart Research Assistant")
st.caption("Ask a question → searched live with Tavily → summarized with Gemini → stored in ChromaDB for follow-ups.")

with st.sidebar:
    st.subheader("Stored research")
    collection = get_collection()
    st.write(f"{collection.count()} research entries stored this session")
    if st.button("Clear all stored research"):
        ids = collection.get()["ids"]
        if ids:
            collection.delete(ids=ids)
        st.session_state.history = []
        st.rerun()

tab1, tab2 = st.tabs(["New research", "Ask a follow-up"])

with tab1:
    query = st.text_input("What do you want to research?", key="research_query")
    if st.button("Research", type="primary") and query:
        with st.spinner("Searching the web..."):
            results = web_search(query)
        if not results:
            st.warning("No search results found.")
        else:
            with st.spinner("Summarizing with Gemini..."):
                summary = summarize_with_gemini(query, results)
            sources = [r.get("url", "") for r in results]
            store_research(query, summary, sources)
            st.session_state.history.append({"type": "research", "query": query, "summary": summary, "sources": sources})

    for item in reversed(st.session_state.history):
        if item["type"] == "research":
            with st.chat_message("assistant"):
                st.markdown(f"**{item['query']}**")
                st.write(item["summary"])
                with st.expander("Sources"):
                    for s in item["sources"]:
                        st.write(s)

with tab2:
    st.write("Ask a question based on everything you've researched so far in this session.")
    followup = st.text_input("Your follow-up question", key="followup_query")
    if st.button("Ask", type="primary") and followup:
        with st.spinner("Thinking..."):
            answer = answer_followup(followup)
        st.session_state.history.append({"type": "followup", "question": followup, "answer": answer})

    for item in reversed(st.session_state.history):
        if item["type"] == "followup":
            with st.chat_message("user"):
                st.write(item["question"])
            with st.chat_message("assistant"):
                st.write(item["answer"])

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess, time

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1", shell=True)
time.sleep(8)

!grep -o 'https://[a-zA-Z0-9.-]*\.trycloudflare\.com' cloudflared.log | head -n 1